In [2]:
import pandas as pd
import numpy as np

import joblib
# import pickle

In [3]:
df = pd.read_csv('hotel_bookings.csv')

In [4]:
df_num = df.select_dtypes(include=['number'])
df_cat = df.select_dtypes(include=['object', 'category'])

In [5]:
df_num

,is_canceled,lead_time,arrival_date_week_number,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,booking_changes,required_car_parking_spaces,total_of_special_requests
0,1,35,13,0,1,2,0.0,0,0,0,1,0,0
1,0,0,19,0,1,1,0.0,0,0,0,0,0,0
2,0,1,29,0,1,2,0.0,0,1,0,0,1,0
3,1,378,31,4,10,2,0.0,0,0,0,0,0,0
4,0,173,31,1,2,2,1.0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,1,0,15,0,0,0,0.0,0,0,0,0,0,0
119386,1,0,15,0,0,0,0.0,0,0,0,0,0,0
119387,1,0,15,0,0,0,0.0,0,0,0,0,0,0
119388,1,0,32,0,0,0,0.0,0,0,0,0,0,0


In [4]:
# encoding categorical variables

df_cat['agent'] = df_cat['agent'].map({'Yes' : 0, 'No' : 1})

df_cat['company'] = df_cat['company'].map({'Yes' : 0, 'No' : 1})

df_cat['hotel'] = df_cat['hotel'].map({'Resort Hotel' : 0, 'City Hotel' : 1})

df_cat['meal'] = df_cat['meal'].map({'BB' : 0, 'FB': 1, 'HB': 2, 'SC': 3, 'Undefined': 4})

df_cat['market_segment'] = df_cat['market_segment'].map({'Direct': 0, 'Corporate': 1, 'Online TA': 2, 'Offline TA/TO': 3,
                                                           'Complementary': 4, 'Groups': 5, 'Undefined': 6, 'Aviation': 7})

df_cat['arrival_date_month'] = df_cat['arrival_date_month'].map({'January': 0, 'February': 1, 'March': 2, 'April': 3,
                                'May': 4, 'June': 5, 'July': 6, 'August': 7, 'September': 8, 'October': 9, 'November': 10, 'December': 11})

df_cat['reserved_room_type'] = df_cat['reserved_room_type'].map({'Single': 0, 'Double': 1, 'Triple': 2, 'Studio': 3})

df_cat['deposit_type'] = df_cat['deposit_type'].map({'No Deposit': 0, 'Refundable': 1, 'Non Refund': 2})

df_cat['customer_type'] = df_cat['customer_type'].map({'Transient': 0, 'Contract': 1, 'Transient-Party': 2, 'Group': 3})

In [5]:
df_merged = pd.concat([df_num, df_cat], axis = 1)

In [ ]:
columns_to_drop = ['company', 'is_repeated_guest', 'adults', 'reserved_room_type', 'babies', 'stays_in_week_nights', 'meal', 'arrival_date_month', 
                   'arrival_date_week_number', 'children', 'stays_in_weekend_nights']
df_merged = df_merged.drop(columns=columns_to_drop)

In [8]:
X = df_merged.iloc[: , 1:]

In [15]:
X

,lead_time,previous_cancellations,booking_changes,required_car_parking_spaces,total_of_special_requests,hotel,market_segment,deposit_type,agent,customer_type
0,35,0,1,0,0,1,3,2,0,0
1,0,0,0,0,0,1,3,0,0,0
2,1,0,0,1,0,0,1,0,1,0
3,378,0,0,0,0,0,2,0,0,0
4,173,0,0,0,1,1,3,0,0,2
...,...,...,...,...,...,...,...,...,...,...
119385,0,0,0,0,0,1,4,0,1,0
119386,0,0,0,0,0,1,4,0,1,0
119387,0,0,0,0,0,1,4,0,1,0
119388,0,0,0,0,0,1,2,0,1,0


In [30]:
headers = X.columns.tolist()
headers

['lead_time',
 'previous_cancellations',
 'booking_changes',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'hotel',
 'market_segment',
 'deposit_type',
 'agent',
 'customer_type']

In [10]:
# Load the model
with open('rd_clf.model', 'rb') as f:
    model = joblib.load(f)

In [ ]:
#with open('cat.pickle', 'rb') as f:
#    model = pickle.load(f)

In [23]:
df_pred = model.predict(X)
df_pred1 = pd.DataFrame()
df_pred1['Predicted'] = df_pred.data.tolist()
df_real =df
df_final = pd.concat([df_pred1, df_real], axis = 1)
df_final

,Predicted,is_canceled,hotel,lead_time,arrival_date_month,arrival_date_week_number,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,is_repeated_guest,previous_cancellations,reserved_room_type,booking_changes,deposit_type,agent,company,customer_type,required_car_parking_spaces,total_of_special_requests
0,1,1,City Hotel,35,March,13,0,1,2,0.0,...,0,0,Single,1,Non Refund,Yes,No,Transient,0,0
1,0,0,City Hotel,0,May,19,0,1,1,0.0,...,0,0,Single,0,No Deposit,Yes,No,Transient,0,0
2,0,0,Resort Hotel,1,July,29,0,1,2,0.0,...,1,0,Single,0,No Deposit,No,No,Transient,1,0
3,1,1,Resort Hotel,378,August,31,4,10,2,0.0,...,0,0,Single,0,No Deposit,Yes,No,Transient,0,0
4,0,0,City Hotel,173,July,31,1,2,2,1.0,...,0,0,Single,0,No Deposit,Yes,No,Transient-Party,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,0,1,City Hotel,0,April,15,0,0,0,0.0,...,0,0,Triple,0,No Deposit,No,Yes,Transient,0,0
119386,0,1,City Hotel,0,April,15,0,0,0,0.0,...,0,0,Triple,0,No Deposit,No,Yes,Transient,0,0
119387,0,1,City Hotel,0,April,15,0,0,0,0.0,...,0,0,Triple,0,No Deposit,No,Yes,Transient,0,0
119388,0,1,City Hotel,0,August,32,0,0,0,0.0,...,0,0,Triple,0,No Deposit,No,No,Transient,0,0


In [24]:
df_pred = model.predict(X)
df_pred

array([1, 0, 0, ..., 0, 0, 0])

In [13]:
df_pred1

,Predicted
0,1
1,0
2,0
3,1
4,0
...,...
119385,0
119386,0
119387,0
119388,0


In [59]:
df_test1 = pd.read_csv('single_test1.csv')

In [60]:
df_test1


,is_canceled,hotel,lead_time,market_segment,previous_cancellations,booking_changes,deposit_type,agent,customer_type,required_car_parking_spaces,total_of_special_requests
0,1,City Hotel,35,Offline TA/TO,1,1,Non Refund,Yes,Transient,1,1


In [61]:
df_test1['hotel'] = df_test1['hotel'].map({'Resort Hotel' : 0, 'City Hotel' : 1})
df_test1['market_segment'] = df_test1['market_segment'].map({'Direct': 0, 'Corporate': 1, 'Online TA': 2, 'Offline TA/TO': 3,
                                                           'Complementary': 4, 'Groups': 5, 'Undefined': 6, 'Aviation': 7})
df_test1['agent'] = df_test1['agent'].map({'Yes' : 0, 'No' : 1})
df_test1['deposit_type'] = df_test1['deposit_type'].map({'No Deposit': 0, 'Refundable': 1, 'Non Refund': 2})

df_test1['customer_type'] = df_test1['customer_type'].map({'Transient': 0, 'Contract': 1, 'Transient-Party': 2, 'Group': 3})


In [64]:
df_test1 = df_test1[headers]
df_test1

,lead_time,previous_cancellations,booking_changes,required_car_parking_spaces,total_of_special_requests,hotel,market_segment,deposit_type,agent,customer_type
0,35,1,1,1,1,1,3,2,0,0


In [65]:
df_pred2 = model.predict(df_test1)
df_pred2

array([1])